# 💻 AstroLoc-ML · Local Training

Train end-to-end on your own machine. Designed for Apple Silicon (MPS) — also works on
CUDA or CPU.

**Why local over Colab:** no disconnects, no Drive setup, multi-core CPU for the renderer,
your trained weights stay on disk.

**Before you start:**
1. From the project root, activate the venv:
   ```bash
   source .venv/bin/activate
   pip install jupyter ipykernel
   python -m ipykernel install --user --name=astroloc --display-name='AstroLoc (.venv)'
   jupyter notebook notebooks/local_train.ipynb
   ```
2. In Jupyter: **Kernel → Change kernel → AstroLoc (.venv)**.
3. Run cells top to bottom (or `Cell → Run All`).

## 1 · Device + environment check

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
%cd {ROOT}

import torch
print(f'torch={torch.__version__}')
print(f'cuda_available={torch.cuda.is_available()}')
print(f'mps_available={torch.backends.mps.is_available()}')

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'\nUsing device: {DEVICE}')

## 2 · Verify the HYG catalog is on disk

Should already be there from the previous run. If not, the cell downloads it.

In [ ]:
import pathlib, subprocess
cat_path = pathlib.Path('data/catalogs/hygdata_v3.csv')
cat_path.parent.mkdir(parents=True, exist_ok=True)
if cat_path.exists() and cat_path.stat().st_size > 1_000_000:
    print(f'✅ HYG present: {cat_path.stat().st_size/1e6:.1f} MB')
else:
    print('Downloading HYG v41 (~32 MB)...')
    subprocess.run(['curl', '-sL', '-o', str(cat_path),
                    'https://raw.githubusercontent.com/astronexus/HYG-Database/main/hyg/CURRENT/hygdata_v41.csv'],
                   check=True)
    print(f'✅ Downloaded: {cat_path.stat().st_size/1e6:.1f} MB')

## 3 · Pipeline sanity check

Loads catalog, renders a sample, runs a forward pass on `DEVICE`. ~5 s.

In [ ]:
import time, numpy as np, torch
from src.data.catalog import load_hyg_catalog
from src.data.renderer import render_star_field
from src.models.astrolocnet import AstroLocNet

catalog = load_hyg_catalog('data/catalogs/hygdata_v3.csv', mag_limit=8.0)
print(f'Loaded {len(catalog):,} stars')

t = time.perf_counter()
img = render_star_field(85.0, -2.0, 30.0, 0.0, catalog, image_size=224,
                        rng=np.random.default_rng(0))
print(f'Render: {(time.perf_counter()-t)*1000:.0f} ms · shape={img.shape}')

model = AstroLocNet(pretrained=False).to(DEVICE)
x = torch.randn(2, 3, 224, 224, device=DEVICE)
y = model(x)
print(f'Forward pass on {DEVICE}: in={tuple(x.shape)} out={tuple(y.shape)}')

## 4 · Benchmark the renderer (informs training-time estimate)

In [ ]:
import time, numpy as np
# Warm up
for _ in range(3):
    render_star_field(0, 0, 30, 0, catalog, image_size=224, rng=np.random.default_rng(0))

N = 50
t = time.perf_counter()
for i in range(N):
    render_star_field(np.random.uniform(0, 360), np.random.uniform(-60, 60),
                      np.random.uniform(15, 80), 0, catalog,
                      image_size=224, rng=np.random.default_rng(i))
dt = (time.perf_counter() - t) / N * 1000
print(f'{dt:.1f} ms / sample (single-threaded)\n')
print('Approximate wall-clock with 6 DataLoader workers:')
for name, n_samples, n_epochs in [('Fast', 5000, 9), ('Standard', 20000, 15), ('Full', 50000, 20)]:
    total = n_samples * n_epochs
    parallel_min = total * (dt/1000) / 60 / 6
    print(f'  {name:<8} {n_samples:>6,} samples × {n_epochs} ep = {total:>7,} renders → ~{parallel_min:>5.1f} min')

## 5 · (Optional) smoke training — ~30 s

Skip if you've done it before.

In [ ]:
!.venv/bin/python -u train.py --config configs/default.yaml --smoke --skip-phase3 --device {DEVICE}

## 6 · Real training run

Pick one of the presets below. CPU-bound on the renderer (model is fast on MPS/CUDA).

| Preset       | `--train-samples` | Epochs (p1/p2) | M-series wall-clock |
| ------------ | ----------------- | -------------- | ------------------- |
| Fast         | 5,000             | 3 / 6          | ~8–12 min           |
| **Standard** | 20,000            | 5 / 10         | **~40–60 min**      |
| Full         | 50,000            | 5 / 15         | ~2–3 hours          |

5K samples plateaus around ~65° val sep (barely learned). To actually break into the
single-digit-degree range you really want at least 20K samples + 15 epochs.

The `-u` flag is important — unbuffered stdout means you see streaming JSON live.

In [ ]:
# Standard preset (recommended) — ~40-60 min on M-series.
!.venv/bin/python -u train.py --config configs/default.yaml --device {DEVICE} --skip-phase3 \
  --train-samples 20000 --val-samples 1000 --num-workers 6 \
  --epochs-phase1 5 --epochs-phase2 10

# === Fast preset (~8-12 min) — uncomment if you want a quick run ===
# !.venv/bin/python -u train.py --config configs/default.yaml --device {DEVICE} --skip-phase3 \
#   --train-samples 5000 --val-samples 500 --num-workers 6 \
#   --epochs-phase1 3 --epochs-phase2 6

# === Full schedule (~2-3 h) — only if you really want to push it ===
# !.venv/bin/python -u train.py --config configs/default.yaml --device {DEVICE} --skip-phase3 \
#   --num-workers 6

## 7 · Inspect the trained checkpoint

`last_phase2.pt` carries the full multi-phase history; `best.pt` carries the snapshot at
the lowest val angular separation.

In [ ]:
import torch
import pandas as pd

for name in ['best.pt', 'last_phase1.pt', 'last_phase2.pt']:
    p = pathlib.Path('checkpoints') / name
    if p.exists():
        c = torch.load(p, map_location='cpu', weights_only=False)
        print(f'{name}: best_metric={c["best_metric"]:.3f}°  epochs={len(c.get("history", []))}')

ckpt = torch.load('checkpoints/last_phase2.pt', map_location='cpu', weights_only=False)
rows = []
for h in ckpt['history']:
    v = h['val']
    rows.append({'phase': h['phase'], 'epoch': h['epoch'],
                 'train_loss': h['train']['loss'], 'val_loss': v['loss'],
                 'val_sep_deg': v['ang_sep_mean_deg'],
                 'val_median_deg': v['ang_sep_median_deg'],
                 'within_5°_%': v['pct_within_5_deg'],
                 'within_1°_%': v['pct_within_1_deg']})
df = pd.DataFrame(rows)
df.round(3)

## 8 · Plot training curves inline

In [ ]:
import matplotlib.pyplot as plt
df['x'] = range(len(df))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(df['x'], df['train_loss'], label='train', marker='o', color='#7c5cff')
axes[0].plot(df['x'], df['val_loss'], label='val', marker='s', color='#ffd166')
axes[0].set_title('Loss across phases'); axes[0].set_xlabel('epoch index'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(df['x'], df['val_sep_deg'], color='#3ad29f', marker='o')
axes[1].set_title('Val angular separation (°)'); axes[1].set_xlabel('epoch index'); axes[1].grid(alpha=0.3)
axes[2].plot(df['x'], df['within_5°_%'], color='#f5a524', marker='o', label='within 5°')
axes[2].plot(df['x'], df['within_1°_%'], color='#f56565', marker='s', label='within 1°')
axes[2].set_title('Tolerance hit rates (%)'); axes[2].set_xlabel('epoch index'); axes[2].legend(); axes[2].grid(alpha=0.3)
for ax in axes:
    for i in range(1, len(df)):
        if df['phase'].iloc[i] != df['phase'].iloc[i-1]:
            ax.axvline(i - 0.5, color='gray', linestyle='--', alpha=0.5)
fig.tight_layout(); plt.show()

## 9 · Regenerate all README figures from the trained model

In [ ]:
!.venv/bin/python scripts/generate_readme_figures.py
!ls -lh reports/figures/

In [ ]:
from IPython.display import Image, display
for p in ['reports/figures/05_training_curves.png',
          'reports/figures/09_demo_overlay_0.png',
          'reports/figures/09_demo_overlay_1.png',
          'reports/figures/09_demo_overlay_2.png']:
    display(Image(p))

## 10 · Test a single prediction

Sanity-check inference: render a known sky pointing, run the model, see how far off it is.

In [ ]:
import numpy as np
from src.inference.predict import load_model, predict_image
from src.data.renderer import render_star_field, sample_random_pointing
from src.utils.coordinates import angular_separation_deg

model = load_model('checkpoints/best.pt', device=DEVICE)
rng = np.random.default_rng(42)
for i in range(5):
    ra, dec, rot, fw = sample_random_pointing(rng, (20.0, 60.0))
    img = render_star_field(ra, dec, fw, rot, catalog, image_size=224, rng=np.random.default_rng(i))
    img_u8 = (img * 255).astype(np.uint8)
    pred = predict_image(model, img_u8, device=DEVICE)
    sep = angular_separation_deg(ra, dec, pred.ra_deg, pred.dec_deg)
    print(f'Truth (RA={ra:6.2f}°, Dec={dec:+6.2f}°)  '
          f'Pred (RA={pred.ra_deg:6.2f}°, Dec={pred.dec_deg:+6.2f}°)  '
          f'Δ={sep:6.2f}°')

---
## Next steps

- **5K-sample model plateaued around 65°.** That's expected: not enough data to fit a 5M-parameter network.
- **20K-sample standard run typically hits 8–15°** mean angular separation by epoch 10.
- **50K-sample full run** can break under 5° with enough patience.
- For real-world generalization, you also want **Phase 3 fine-tuning on Astrometry.net solved images**
  (drop them into `data/real_images/` first, then set `real_data.enabled: true` in `configs/default.yaml`).
- Inspect ablation vs the classical solver in `notebooks/07_ablation_study.ipynb` after training improves.